In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as ss
from scipy import stats

from neuromaps import nulls
from neuromaps.images import dlabel_to_gifti
from netneurotools import datasets as nntdata

plt.rcParams.update({'font.size': 14})

## Schaefer-600 parcellation (for spatial null models)

In [ ]:
# Schaefer 600-parcel, 7-network parcellation in fsLR space, used only to build spatial
# null models below (`parcellation` = the (lh, rh) GIFTI label images). CMrGlu/CMrO2 have
# already been parcellated and merged with primary power/current dipole by
# calculate_parcellate_primary_p/build_pri_power_met_csv.py, whose output this notebook
# reads directly below -- no need to re-fetch/re-parcellate those annotations here.
schaefer = nntdata.fetch_schaefer2018('fslr32k')['600Parcels7Networks']
parcellation = dlabel_to_gifti(schaefer)

## Load primary power/current-dipole data (32-subject cohort) merged with CMrGlu/CMrO2 metabolism

In [ ]:
# Uses PRIMARY dipole power/current: I_dip^2 * cortical-column resistance
# (calculate_fem_column_power.m), parcellated and subject-averaged across the full
# 32-subject cohort (parcellate_central_surface_s600.m), already merged with CMrGlu/CMrO2
# by build_pri_power_met_csv.py. This is the main data source for the whole notebook --
# see the very end for an optional secondary-power toggle.
MERGED_CSV = '/export02/data/vikramn/hbm_manuscript_code/outputs/primary_p/cent_surf_fem_fwd/rms_2min_cent_pri_power_met_broad.csv'
output_dir = '/export02/data/vikramn/hbm_manuscript_code/outputs/primary_p/figures/'

meg_met = pd.read_csv(MERGED_CSV)
meg_rgn_p = meg_met[['region', 'mean_p', 'cmrglu', 'cmro2']].rename(columns={'mean_p': 'mean'})
meg_rgn_i = meg_met[['region', 'mean_i', 'cmrglu', 'cmro2']].rename(columns={'mean_i': 'mean'})

## Load gene-expression-derived metabolism data (Pourmajidian et al., 2025)

In [ ]:
# The pickle indexes each of its 34 process Series by integer Schaefer-600 label (1-600, the
# neuromaps dlabel.nii numbering) rather than by region name, so it needs
# neuromaps_s600_7net_rgns.txt (label k -> rownames[k-1]) to translate into the same
# "Cont_Cing_1 L"-style region names the primary-power pipeline uses -- confirmed to already
# match directly (same convention build_pri_power_met_csv.py's cmrglu/cmro2 merge relies on).
PICKLE_PATH = '/export02/data/vikramn/hbm_manuscript_code/metabolism_correlations/energy_mean_expression_600.pickle'
ROWNAMES_PATH = '/export02/data/vikramn/baillet_lab_windows/neuromaps_s600_7net_rgns.txt'

with open(ROWNAMES_PATH) as f:
    rownames = [r.strip() for r in f.readlines()]

# A duplicated name in rownames.txt (with some other region silently missing to compensate)
# is invisible to a simple pre/post row-count check below: the duplicate fans out to an
# extra merged row for one region while the missing region drops its own row, netting to
# the same total count but silently corrupting *which* metabolism values got paired with
# which power/current values. Caught exactly this once (line 600 duplicated line 587,
# 'Default_pCunPCC_14 R' silently missing) -- fail loudly instead of merging past it.
dupes = pd.Series(rownames)[pd.Series(rownames).duplicated(keep=False)].unique().tolist()
if dupes:
    raise ValueError(f'neuromaps_s600_7net_rgns.txt has duplicate region name(s): {dupes} '
                      f'-- fix the source file before merging (a duplicate silently drops '
                      f'some other region instead of erroring, since {len(rownames)} names '
                      f'total still looks right).')

s600_met_data = pd.read_pickle(PICKLE_PATH)
process_cols = list(s600_met_data.keys())
met_df = pd.DataFrame({proc: series.sort_index().values for proc, series in s600_met_data.items()})
met_df.insert(0, 'region', rownames)

n_before = len(meg_rgn_p)
meg_rgn_p = meg_rgn_p.merge(met_df, on='region', how='inner')
meg_rgn_i = meg_rgn_i.merge(met_df, on='region', how='inner')
if len(meg_rgn_p) != n_before:
    print(f'WARNING: {n_before - len(meg_rgn_p)}/{n_before} region(s) had no gene-expression '
          f'match and were dropped.')

## Spatial null models (Schaefer-600) for correlation p-values

Instead of the standard parametric formula (a Student-t transform of Spearman's rho, which
assumes each of the 600 parcels is an independent sample), every p-value below is computed
from spatial-autocorrelation-preserving null maps generated by rotating the Schaefer-600
parcellation on the cortical sphere (Alexander-Bloch et al., 2018 "spin test"; see the
[neuromaps null models guide](https://netneurolab.github.io/neuromaps/user_guide/nulls.html)).

For network-level bars, the *same* whole-brain spin rotations are simply restricted to the
parcels belonging to that network rather than being regenerated per network -- a spin is a
valid spatial permutation of any subset of parcels, since it permutes parcel identities
consistently across the whole brain.

In [ ]:
N_PERM = 10000
SEED = 1234


def make_nulls(data):
    """Schaefer-600 spatial null maps for a whole-brain parcellated annotation."""
    return nulls.alexander_bloch(np.asarray(data), atlas='fsLR', density='32k',
                                  parcellation=parcellation, n_perm=N_PERM, seed=SEED)


def spin_pvalue(x, y, y_nulls, indices=None, alpha=0.05):
    """
    Spearman correlation between x and y, with a p-value and |rho| significance threshold
    derived from Schaefer-600 spatial null models (rather than the parametric Student-t
    approximation).

    y_nulls: whole-brain null maps for y, from make_nulls(y) -- shape (600, N_PERM).
    indices: optional subset of parcel indices (e.g. one Yeo network) to restrict the test
             to; the whole-brain null maps are reused rather than regenerated per subset.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    if indices is not None:
        x = x[indices]
        y = y[indices]
        y_nulls = y_nulls[indices, :]

    rho = ss.spearmanr(x, y)[0]
    null_rhos = np.array([ss.spearmanr(x, y_nulls[:, k])[0] for k in range(y_nulls.shape[1])])

    n_perm = len(null_rhos)
    p_spin = (np.sum(np.abs(null_rhos) >= np.abs(rho)) + 1) / (n_perm + 1)
    rho_crit = np.nanpercentile(np.abs(null_rhos), 100 * (1 - alpha))
    return rho, p_spin, rho_crit

## Barplot: top 5 metabolic processes correlated with Power/Current Dipole (whole-brain)

In [ ]:
alpha = 0.05
m = len(process_cols)  # Bonferroni across all metabolism processes tested
alpha_bonf_proc = alpha / m

records = []
for proc in process_cols:
    proc_data = meg_rgn_p[proc].values
    proc_nulls = make_nulls(proc_data)
    rho_p, p_p, crit_p = spin_pvalue(meg_rgn_p['mean'].values, proc_data, proc_nulls, alpha=alpha_bonf_proc)
    rho_i, p_i, crit_i = spin_pvalue(meg_rgn_i['mean'].values, proc_data, proc_nulls, alpha=alpha_bonf_proc)
    records.append((proc, rho_p, p_p, crit_p, rho_i, p_i, crit_i))

df = (pd.DataFrame(records, columns=["process", "rho_power", "p_power", "rho_crit_power",
                                      "rho_dipole", "p_dipole", "rho_crit_dipole"])
        .set_index("process").dropna())

top5_power = df.reindex(df["rho_power"].abs().nlargest(5).index)
top5_dipole = df.reindex(df["rho_dipole"].abs().nlargest(5).index)


def plot_panel(ax, subdf, title, rank_col, crit_col):
    order = subdf[rank_col].abs().sort_values(ascending=False).index
    subdf = subdf.loc[order]

    abs_power = subdf["rho_power"].abs().values
    abs_dipole = subdf["rho_dipole"].abs().values
    sign_power = np.where(subdf["rho_power"].values >= 0, "+", "-")
    sign_dipole = np.where(subdf["rho_dipole"].values >= 0, "+", "-")

    x = np.arange(len(subdf))
    width = 0.38
    bp = ax.bar(x - width / 2, abs_power, width, label="Power (p)")
    bi = ax.bar(x + width / 2, abs_dipole, width, label="Dipole (i)")

    # Each process is an independent comparison -- draw its Bonferroni threshold as its own
    # short segment rather than one connected line across processes (which would visually
    # imply a trend between categorically unrelated bars).
    crit_vals = subdf[crit_col].values
    for xi, thr in enumerate(crit_vals):
        ax.plot([xi - width, xi + width], [thr, thr], linestyle=":", color="black",
                label=f"Spin-test Bonf. m={m}" if xi == 0 else None)

    for bars, signs in [(bp, sign_power), (bi, sign_dipole)]:
        for r, s in zip(bars, signs):
            h = r.get_height()
            ax.text(r.get_x() + r.get_width() / 2, h + 0.015, s, ha="center", va="bottom", fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=20, ha="right")
    ax.set_ylim(0, max(abs_power.max(), abs_dipole.max(), crit_vals.max()) * 1.25)
    ax.set_ylabel("Spearman |rho|")
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
plot_panel(axes[0], top5_power, "Top 5 Correlated Processes With Power", "rho_power", "rho_crit_power")
plot_panel(axes[1], top5_dipole, "Top 5 Correlated Processes With Current Dipole", "rho_dipole", "rho_crit_dipole")

handles, labels = [], []
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    handles += h
    labels += l
uniq = dict(zip(labels, handles))
fig.legend(uniq.values(), uniq.keys(), loc="upper center", ncol=3, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig(output_dir + 'top5_processes_power_dipole.png', transparent=False)
plt.show()

# Printed here for direct reference/citation: Spearman correlations and spin-test p-values
# (Schaefer-600 spatial nulls, N_PERM=10000), flagged against the Bonferroni-corrected alpha
# (0.05 / 34 processes = 0.00147). A p-value of 0.0001 is the empirical floor achievable with
# N_PERM permutations (1/(N_PERM+1)) -- it means "0/N_PERM null rotations were as extreme",
# not literally "p < 0.0001"; report it as such rather than as an exact figure.
print(f"==== Top 5 Power-ranked Processes (spin-test p-values, Schaefer-600; Bonferroni alpha={alpha_bonf_proc:.5f}) ====")
for proc, row in top5_power.iterrows():
    sig_p = "*" if row['p_power'] < alpha_bonf_proc else " "
    sig_i = "*" if row['p_dipole'] < alpha_bonf_proc else " "
    print(f"{proc:20s} | Power: rho={row['rho_power']:+.3f}, p_spin={row['p_power']:.4f}{sig_p} "
          f"| Dipole: rho={row['rho_dipole']:+.3f}, p_spin={row['p_dipole']:.4f}{sig_i}")

print(f"\n==== Top 5 Dipole-ranked Processes (spin-test p-values, Schaefer-600; Bonferroni alpha={alpha_bonf_proc:.5f}) ====")
for proc, row in top5_dipole.iterrows():
    sig_p = "*" if row['p_power'] < alpha_bonf_proc else " "
    sig_i = "*" if row['p_dipole'] < alpha_bonf_proc else " "
    print(f"{proc:20s} | Power: rho={row['rho_power']:+.3f}, p_spin={row['p_power']:.4f}{sig_p} "
          f"| Dipole: rho={row['rho_dipole']:+.3f}, p_spin={row['p_dipole']:.4f}{sig_i}")
print("(* = significant at Bonferroni-corrected alpha)")

## Supplementary Table 1: whole-brain correlations, all 34 metabolic processes

In [ ]:
# Whole-brain correlations between MEG-derived current/power and every gene-expression-derived
# metabolic process from Pourmajidian et al., 2025 (full pathway names in their Supplementary
# Figure S9). Sorted by |Power correlation| descending, matching the ranking used for the
# top-5 barplot above -- built from the same `df` computed there.
supp_table1 = df[['rho_power', 'rho_dipole']].rename(
    columns={'rho_power': 'Power correlation', 'rho_dipole': 'Current dipole correlation'}
)
supp_table1 = supp_table1.reindex(supp_table1['Power correlation'].abs().sort_values(ascending=False).index)
supp_table1.index.name = 'Pathway'

supp_table1.round(4).to_csv(output_dir + 'supplementary_table1_process_correlations.csv')
print(supp_table1.round(4).to_string())

## Auto-select the 2 processes most correlated with primary power

Replaces the previously hardcoded kb_util/kb_metabolism pair: whichever two
gene-expression-derived metabolic processes correlate most strongly (by |Spearman rho|,
whole-brain) with primary power (`mean_p`/`primary_dipole_p`) are used for the per-network
barplot below.

In [ ]:
top2_processes = df["rho_power"].abs().nlargest(2).index.tolist()
print("Auto-selected top-2 processes (most correlated with primary power):", top2_processes)

## Yeo-7-network definitions

In [ ]:
yeo_sets = [
    list(range(0, 82)),
    list(range(82, 212)),
    list(range(212, 284)),
    list(range(284, 326)),
    list(range(326, 399)),
    list(range(399, 511)),
    list(range(511, 600)),
]

yeo_nets = ['Cont', 'Default', 'DorsAttn', 'Limbic', 'SalVentAttn', 'SomMot', 'Vis']

alpha_bonf = 0.05 / len(yeo_nets)  # Bonferroni across seven networks


def netwise_spin_stats(meg_df, met_col, met_nulls, alpha):
    """Per-network Spearman rho, spin-test p-value, and |rho| threshold, reusing the
    same whole-brain spatial nulls restricted to each network's parcels."""
    rhos, pvals, rho_crits = [], [], []
    x_full = meg_df['mean'].values
    y_full = meg_df[met_col].values
    for idx in yeo_sets:
        rho, p_val, rho_crit = spin_pvalue(x_full, y_full, met_nulls, indices=idx, alpha=alpha)
        rhos.append(rho)
        pvals.append(p_val)
        rho_crits.append(rho_crit)
    return np.array(rhos), np.array(pvals), np.array(rho_crits)

## Barplot: auto-selected processes vs Power/Current Dipole, by Yeo network

In [ ]:
def plot_process_panel_both(ax, met_col, title):
    met_nulls = make_nulls(meg_rgn_p[met_col].values)  # same metabolism map used for both p and i

    r_p, p_p, crit_p_bonf = netwise_spin_stats(meg_rgn_p, met_col, met_nulls, alpha_bonf)
    r_i, p_i, crit_i_bonf = netwise_spin_stats(meg_rgn_i, met_col, met_nulls, alpha_bonf)

    abs_p, abs_i = np.abs(r_p), np.abs(r_i)
    sign_p = np.where(r_p >= 0, "+", "-")
    sign_i = np.where(r_i >= 0, "+", "-")

    x = np.arange(len(yeo_nets))
    width = 0.38
    bp = ax.bar(x - width / 2, abs_p, width, label="Power (p)")
    bi = ax.bar(x + width / 2, abs_i, width, label="Dipole (i)")

    rho_bonf_line = np.maximum(crit_p_bonf, crit_i_bonf)

    for xi, thr in enumerate(rho_bonf_line):
        ax.plot([xi - width, xi + width], [thr, thr], linestyle=":", linewidth=1.5, color="black",
                label="Spin-test Bonferroni alpha/7" if xi == 0 else None)

    for bars, signs in [(bp, sign_p), (bi, sign_i)]:
        for r, s in zip(bars, signs):
            h = r.get_height()
            ax.text(r.get_x() + r.get_width() / 2, h + 0.015, s, ha="center", va="bottom", fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(yeo_nets, rotation=30, ha="right", fontsize=11)
    ax.set_ylim(0, max(abs_p.max(), abs_i.max(), rho_bonf_line.max()) * 1.25)
    ax.set_ylabel("Spearman |rho|")
    ax.set_title(title)

    return r_p, p_p, r_i, p_i


fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
r_p_proc1, p_p_proc1, r_i_proc1, p_i_proc1 = plot_process_panel_both(axes[0], top2_processes[0], f"{top2_processes[0]} vs Calculated Power/Current Dipole")
r_p_proc2, p_p_proc2, r_i_proc2, p_i_proc2 = plot_process_panel_both(axes[1], top2_processes[1], f"{top2_processes[1]} vs Calculated Power/Current Dipole")

handles, labels = [], []
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    handles += h
    labels += l
uniq = dict(zip(labels, handles))
fig.legend(uniq.values(), uniq.keys(), loc="upper center", ncol=3, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig(output_dir + 'top2_processes_by_network.png', transparent=False)
plt.show()

# Printed here for direct reference/citation: Spearman correlations and spin-test p-values
# (Schaefer-600 spatial nulls restricted to each network's parcels, N_PERM=10000), flagged
# against the Bonferroni-corrected alpha (0.05 / 7 networks = 0.00714).
def print_netwise_processes(label, r_p, p_p, r_i, p_i):
    print(f"==== {label} vs MEG (per network, spin-test p-values; Bonferroni alpha={alpha_bonf:.5f}) ====")
    for net, rp, pp, ri, pi in zip(yeo_nets, r_p, p_p, r_i, p_i):
        sig_p = "*" if pp < alpha_bonf else " "
        sig_i = "*" if pi < alpha_bonf else " "
        print(f"{net:12s} | Power: rho={rp:+.3f}, p_spin={pp:.4f}{sig_p} "
              f"| Dipole: rho={ri:+.3f}, p_spin={pi:.4f}{sig_i}")


print_netwise_processes(top2_processes[0], r_p_proc1, p_p_proc1, r_i_proc1, p_i_proc1)
print()
print_netwise_processes(top2_processes[1], r_p_proc2, p_p_proc2, r_i_proc2, p_i_proc2)
print("(* = significant at Bonferroni-corrected alpha)")

## Barplot: CMrGlu/CMrO2 vs Power/Current Dipole (whole-brain)

In [ ]:
alpha = 0.05
processes = ["cmrglu", "cmro2"]
m = len(processes)
alpha_bonf_met = alpha / m

records = []
for met in processes:
    met_nulls = make_nulls(meg_rgn_p[met].values)
    rho_p, p_p, crit_p = spin_pvalue(meg_rgn_p['mean'].values, meg_rgn_p[met].values, met_nulls, alpha=alpha_bonf_met)
    rho_i, p_i, crit_i = spin_pvalue(meg_rgn_i['mean'].values, meg_rgn_i[met].values, met_nulls, alpha=alpha_bonf_met)
    records.append((met, rho_p, p_p, crit_p, rho_i, p_i, crit_i))

df_glu_o2 = (pd.DataFrame(records, columns=["process", "rho_power", "p_power", "rho_crit_power",
                                             "rho_dipole", "p_dipole", "rho_crit_dipole"])
               .set_index("process"))

abs_power = df_glu_o2["rho_power"].abs().values
abs_dipole = df_glu_o2["rho_dipole"].abs().values
sign_power = np.where(df_glu_o2["rho_power"].values >= 0, "+", "-")
sign_dipole = np.where(df_glu_o2["rho_dipole"].values >= 0, "+", "-")
rho_crit_line = np.maximum(df_glu_o2["rho_crit_power"].values, df_glu_o2["rho_crit_dipole"].values)

x = np.arange(len(processes))
width = 0.38

fig, ax = plt.subplots(figsize=(8, 6))
bp = ax.bar(x - width / 2, abs_power, width, label="Power (p)")
bi = ax.bar(x + width / 2, abs_dipole, width, label="Dipole (i)")

# CMrGlu and CMrO2 are independent comparisons -- draw each one's Bonferroni threshold as
# its own short segment rather than one connected line across both (which would visually
# imply a trend between two unrelated bars).
for xi, thr in enumerate(rho_crit_line):
    ax.plot([xi - width, xi + width], [thr, thr], linestyle=":", linewidth=1.5, color="black",
            label=f"Spin-test Bonf. m={m}" if xi == 0 else None)

for bars, signs, vals in [(bp, sign_power, abs_power), (bi, sign_dipole, abs_dipole)]:
    for r, s, v in zip(bars, signs, vals):
        h = r.get_height()
        ax.text(r.get_x() + r.get_width() / 2, h + 0.02, f"{s}{v:.2f}", ha="center", va="bottom", fontsize=13)

ax.set_xticks(x)
ax.set_xticklabels(["CMrGlu", "CMrO2"])
ax.set_ylabel("Spearman |rho|")
ax.set_ylim(0, max(abs_power.max(), abs_dipole.max(), rho_crit_line.max()) * 1.25)
ax.set_title("Correlation of Power/Current Dipole with CMrGlu and CMrO2")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig(output_dir + 'cmrglu_cmro2_whole_brain.png', transparent=False)
plt.show()

# Printed here for direct reference/citation: Spearman correlations and spin-test p-values
# (Schaefer-600 spatial nulls, N_PERM=10000), flagged against the Bonferroni-corrected alpha
# (0.05 / 2 = 0.025).
print(f"==== Correlations and spin-test p-values (Schaefer-600 nulls; Bonferroni alpha={alpha_bonf_met:.5f}) ====")
for proc, row in df_glu_o2.iterrows():
    sig_p = "*" if row['p_power'] < alpha_bonf_met else " "
    sig_i = "*" if row['p_dipole'] < alpha_bonf_met else " "
    print(f"{proc:7s} | Power: rho={row['rho_power']:+.3f}, p_spin={row['p_power']:.4f}{sig_p} "
          f"| Dipole: rho={row['rho_dipole']:+.3f}, p_spin={row['p_dipole']:.4f}{sig_i}")
print("(* = significant at Bonferroni-corrected alpha)")

## Barplot: CMrGlu/CMrO2 vs Power/Current Dipole, by Yeo network

In [ ]:
def plot_met_panel_both(ax, met_col, title):
    met_nulls = make_nulls(meg_rgn_p[met_col].values)

    r_p, p_p, crit_p_bonf = netwise_spin_stats(meg_rgn_p, met_col, met_nulls, alpha_bonf)
    r_i, p_i, crit_i_bonf = netwise_spin_stats(meg_rgn_i, met_col, met_nulls, alpha_bonf)

    abs_p, abs_i = np.abs(r_p), np.abs(r_i)
    sign_p = np.where(r_p >= 0, "+", "-")
    sign_i = np.where(r_i >= 0, "+", "-")

    x = np.arange(len(yeo_nets))
    width = 0.38
    bp = ax.bar(x - width / 2, abs_p, width, label="Power (p)")
    bi = ax.bar(x + width / 2, abs_i, width, label="Dipole (i)")

    rho_bonf_line = np.maximum(crit_p_bonf, crit_i_bonf)
    for xi, thr in enumerate(rho_bonf_line):
        ax.plot([xi - width, xi + width], [thr, thr], linestyle=":", linewidth=1.5, color="black",
                label="Spin-test Bonferroni alpha/7" if xi == 0 else None)

    for bars, signs in [(bp, sign_p), (bi, sign_i)]:
        for r, s in zip(bars, signs):
            h = r.get_height()
            ax.text(r.get_x() + r.get_width() / 2, h + 0.015, s, ha="center", va="bottom", fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(yeo_nets, rotation=30, ha="right", fontsize=11)
    ax.set_ylim(0, max(abs_p.max(), abs_i.max(), rho_bonf_line.max()) * 1.25)
    ax.set_ylabel("Spearman |rho|")
    ax.set_title(title)
    return r_p, p_p, r_i, p_i


fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
r_p_glu, p_p_glu, r_i_glu, p_i_glu = plot_met_panel_both(axes[0], "cmrglu", "CMrGlu vs Calculated Power/Current Dipole")
r_p_o2, p_p_o2, r_i_o2, p_i_o2 = plot_met_panel_both(axes[1], "cmro2", "CMrO2 vs Calculated Power/Current Dipole")

handles, labels = [], []
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    handles += h
    labels += l
uniq = dict(zip(labels, handles))
fig.legend(uniq.values(), uniq.keys(), loc="upper center", ncol=3, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig(output_dir + 'cmrglu_cmro2_by_network.png', transparent=False)
plt.show()

# Printed here for direct reference/citation: Spearman correlations and spin-test p-values
# (Schaefer-600 spatial nulls restricted to each network's parcels, N_PERM=10000), flagged
# against the Bonferroni-corrected alpha (0.05 / 7 networks = 0.00714).
def print_netwise(label, r_p, p_p, r_i, p_i):
    print(f"==== {label} vs MEG (per network, spin-test p-values; Bonferroni alpha={alpha_bonf:.5f}) ====")
    for net, rp, pp, ri, pi in zip(yeo_nets, r_p, p_p, r_i, p_i):
        sig_p = "*" if pp < alpha_bonf else " "
        sig_i = "*" if pi < alpha_bonf else " "
        print(f"{net:12s} | Power: rho={rp:+.3f}, p_spin={pp:.4f}{sig_p} "
              f"| Dipole: rho={ri:+.3f}, p_spin={pi:.4f}{sig_i}")


print_netwise("CMrGlu", r_p_glu, p_p_glu, r_i_glu, p_i_glu)
print()
print_netwise("CMrO2", r_p_o2, p_p_o2, r_i_o2, p_i_o2)
print("(* = significant at Bonferroni-corrected alpha)")

## Scatterplots: Power/Current Dipole vs. CMrGlu/CMrO2 (supplementary figure)

In [ ]:
name_dict = {'p': 'Power', 'i': 'Current Dipole', 'cmrglu': 'CMrGlu', 'cmro2': 'CMrO2', 'mean': 'Mean'}
unit_dict = {'pmean': 'W', 'imean': 'A.m'}


def generate_meg_met_figs(indices, output_dir, suffix):
    """
    2x2 grid of scatterplots: Power vs CMrGlu, Power vs CMrO2, Current vs CMrGlu, Current vs CMrO2.
    Annotates each panel with the Spearman rho and a spin-test (Schaefer-600 null model) p-value.
    """
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    meg_list = ['p', 'p', 'i', 'i']
    met_list = ['cmrglu', 'cmro2', 'cmrglu', 'cmro2']

    for k in range(4):
        meg = meg_list[k]
        met = met_list[k]
        meg_rgn = meg_rgn_p if meg == 'p' else meg_rgn_i

        x = meg_rgn['mean'].values[indices]
        y_raw = meg_rgn[met].values[indices]
        y = (y_raw - np.mean(y_raw)) / np.std(y_raw)  # z-score for display only

        met_nulls = make_nulls(meg_rgn[met].values)[indices, :]
        rho, p_spin, _ = spin_pvalue(x, y_raw, met_nulls)

        slope, intercept, _, _, _ = stats.linregress(x, y)

        ax = axs[k // 2, k % 2]
        ax.scatter(x, y)
        ax.plot(x, slope * x + intercept, color='red', label='Line of Best Fit')
        ax.set_ylabel(name_dict[met] + ' (scaled)', fontsize=14)
        ax.set_xlabel(name_dict['mean'] + ' in ' + name_dict[meg] + ' per region ' + unit_dict[meg + 'mean'], fontsize=14)
        ax.set_title(suffix + ' Mean ' + name_dict[meg] + ' vs. Mean ' + name_dict[met], fontsize=14)

        textstr = f'Spearman rho: {rho:.2f}, spin-test p: {p_spin:.4f}'
        props = dict(boxstyle='round', facecolor='white', alpha=0.5)
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=14, verticalalignment='top', bbox=props)

    plt.tight_layout()
    plt.savefig(output_dir + 'combined_s600_' + suffix + '.png', transparent=False)
    plt.show()
    plt.close(fig)

In [ ]:
generate_meg_met_figs(indices=np.arange(len(meg_rgn_p)), output_dir=output_dir, suffix='full')

## Supplementary: partial correlations of primary dipole P with PET/transcriptomic measures, controlling for cortical geometry

Per the paper's own derivation (Methods, Eq. 2-4): resistance R = rho\*L/A (rho = avg_gm_res,
a single fixed constant applied uniformly to every region -- `parcellate_cortical_column_res_vol.m`),
and current density I = I_dip/L (I_dip being the primary current dipole moment MEG actually
estimates, L = cortical thickness). Substituting both into P = I^2\*R gives

    P = (I_dip/L)^2 * (rho*L/A) = rho * I_dip^2 / (L*A)

so P's full geometric dependence is **1/(L\*A)**, not R = L/A alone -- squaring the 1/L from
converting I_dip into a current density adds an extra 1/L that combines with R's own L/A to
leave 1/(L\*A). A real concern is therefore that any P-vs-metabolism correlation found above
is actually just a thickness/area-vs-metabolism relationship riding along inside P's own
formula, rather than something P's electrophysiological content (I_dip) adds on top of
geometry alone.

Three separate partial-correlation checks against P, each controlling for a different
geometric confound:
- **thickness alone**
- **area alone**
- **1 / (thickness \* area)** -- the exact combined geometric term that multiplies I_dip^2 in
  P's own formula (up to the fixed constant rho), so this is the correct "remove all of P's
  geometry dependence at once" check, rather than an unconstrained linear combination of
  thickness and area (which wouldn't respect the specific 1/(L\*A) structure P actually has).

Splitting thickness and area into their own single-covariate checks (rather than only the
combined 1/(L\*A) term) lets us see whether one geometric dimension is driving more of the
raw association than the other; the combined check then targets the exact term that actually
appears in P's own formula.

We also report a direct (non-partial) correlation of the current-dipole term I_dip itself
against every target: since P = rho \* I_dip^2 / (L\*A) is exactly I_dip^2 times a purely
geometric factor, and a rank-based partial correlation only removes the best-fit
*linear-in-rank* relationship between P and its geometric covariate (not the exact
multiplicative structure), a geometry-controlled partial correlation of P is only an
approximation of "does I_dip^2 alone (independent of geometry) correlate with metabolism."
I_dip is already directly available (`meg_rgn_i`, the same "Dipole (i)" data plotted in
every figure above), and because I_dip >= 0, Spearman correlation is invariant to the
monotonic square, so correlating I_dip with a target is *exactly* the same test as
correlating I_dip^2 with it -- an exact check, not an approximation. We report it
side-by-side with the geometry-partialled P correlations below.

**Significance thresholds**: PET (cmrglu, cmro2) and gene-expression measures are kept as
two separate Bonferroni families here, matching exactly how the rest of the notebook treats
them -- never pooled -- so this analysis reuses `alpha_bonf_met` (0.05 / 2, the same
threshold as the whole-brain CMrGlu/CMrO2 barplot above) for cmrglu/cmro2, and
`alpha_bonf_proc` (0.05 / 34, the same threshold as the top-5-processes barplot above) for
the 34 gene-expression targets. Pooling all 36 into one combined correction would apply a
stricter threshold to both than either uses elsewhere in the paper, and would misleadingly
imply PET and transcriptomic measures were tested as a single family, which they were not.

In [ ]:
THICK_CSV = '/export02/data/vikramn/hbm_manuscript_code/outputs/s600_subavg_column_thick.csv'
AREA_CSV = '/export02/data/vikramn/hbm_manuscript_code/outputs/s600_subavg_column_sa.csv'

confounds = (pd.read_csv(THICK_CSV).rename(columns={'value': 'thickness'})
             .merge(pd.read_csv(AREA_CSV).rename(columns={'value': 'area'}), on='region'))

n_before = len(meg_rgn_p)
meg_rgn_p_geom = meg_rgn_p.merge(confounds, on='region', how='inner').merge(
    meg_rgn_i[['region', 'mean']].rename(columns={'mean': 'current'}), on='region', how='inner')
if len(meg_rgn_p_geom) != n_before:
    print(f'WARNING: {n_before - len(meg_rgn_p_geom)}/{n_before} region(s) had no thickness/area/'
          f'current match and were dropped.')

power = meg_rgn_p_geom['mean'].values
current = meg_rgn_p_geom['current'].values
thickness = meg_rgn_p_geom['thickness'].values
area = meg_rgn_p_geom['area'].values


def _rank_resid(v, Z):
    """Residuals of rank(v) after an OLS fit on covariate design matrix Z (already includes
    an intercept column)."""
    v_rank = ss.rankdata(v)
    coef = np.linalg.lstsq(Z, v_rank, rcond=None)[0]
    return v_rank - Z @ coef


def partial_spin_pvalue(x, y, y_nulls, z):
    """
    Spearman PARTIAL correlation of x and y controlling for covariate(s) z (n_obs x
    n_covariates, 2D even for a single covariate): rank-transform x, y, and every column of
    z, linearly regress the ranks of x and of y on the ranks of z (with an intercept), then
    Pearson-correlate the two sets of residuals -- with a spin-test p-value computed the same
    way as spin_pvalue, except x and every null column of y are first residualized against
    the SAME z (geometry doesn't change across spin rotations) before correlating.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    z_rank = np.column_stack([ss.rankdata(z[:, j]) for j in range(z.shape[1])])
    Z = np.column_stack([np.ones(len(x)), z_rank])

    x_resid = _rank_resid(x, Z)
    rho = np.corrcoef(x_resid, _rank_resid(y, Z))[0, 1]
    null_rhos = np.array([np.corrcoef(x_resid, _rank_resid(y_nulls[:, k], Z))[0, 1]
                           for k in range(y_nulls.shape[1])])

    n_perm = len(null_rhos)
    p_spin = (np.sum(np.abs(null_rhos) >= np.abs(rho)) + 1) / (n_perm + 1)
    return rho, p_spin


# 'geometry' is 1/(thickness*area), the EXACT combined geometric term that multiplies I_dip^2
# in P's own formula (Methods Eq. 2-4: substituting current density I = I_dip/L into P = I^2*R,
# with R = rho*L/A, gives P = rho*I_dip^2/(L*A)) -- not R = L/A alone, which misses the extra
# 1/L introduced by converting I_dip into a current density before squaring. See the markdown
# cell above for the full derivation.
geometry_covariates = {
    'thickness': thickness[:, None],
    'area': area[:, None],
    'geometry': (1.0 / (thickness * area))[:, None],
}

# Significance thresholds: TWO SEPARATE Bonferroni families, matching exactly how the rest of
# the notebook treats PET vs. gene-expression measures (never pooled) -- alpha_bonf_met
# (already defined above, whole-brain CMrGlu/CMrO2 barplot: 0.05/2) for cmrglu/cmro2, and
# alpha_bonf_proc (already defined above, top-5-processes barplot: 0.05/34) for the
# gene-expression targets. See the markdown cell above for why these are not pooled into one
# combined correction.
targets = ['cmrglu', 'cmro2'] + process_cols
target_alpha = {t: (alpha_bonf_met if t in ('cmrglu', 'cmro2') else alpha_bonf_proc) for t in targets}

partial_records = []
for target in targets:
    target_data = meg_rgn_p_geom[target].values
    target_nulls = make_nulls(target_data)

    rho_raw, p_raw, _ = spin_pvalue(power, target_data, target_nulls, alpha=target_alpha[target])
    # Direct (non-partial) correlation of the current-dipole term I_dip with the same target:
    # since P = rho*I_dip^2/(L*A) and I_dip >= 0, rank(I_dip) == rank(I_dip^2), so this is the
    # exact Spearman correlation of I_dip^2 with the target -- not an approximation -- and
    # tests whether the association survives when reduced to just the current-amplitude
    # component of P, independent of geometry.
    rho_current, p_current, _ = spin_pvalue(current, target_data, target_nulls, alpha=target_alpha[target])
    row = {'target': target, 'rho_raw': rho_raw, 'p_raw': p_raw,
           'rho_current': rho_current, 'p_current': p_current}
    for label, z in geometry_covariates.items():
        rho_p, p_p = partial_spin_pvalue(power, target_data, target_nulls, z)
        row[f'rho_partial_{label}'] = rho_p
        row[f'p_partial_{label}'] = p_p
    partial_records.append(row)

partial_df = pd.DataFrame(partial_records).set_index('target')
partial_df = partial_df.reindex(partial_df['rho_raw'].abs().sort_values(ascending=False).index)
partial_df.round(4).to_csv(output_dir + 'supplementary_partial_correlations_power.csv')

print("==== Partial correlations of primary dipole P with PET/transcriptomic measures ====")
print(f"(spin-test p-values, N_PERM={N_PERM}; significance flagged against TWO SEPARATE Bonferroni "
      f"thresholds matching Fig. 4/5 -- alpha={alpha_bonf_met:.5f} for cmrglu/cmro2 (m=2), "
      f"alpha={alpha_bonf_proc:.5f} for the 34 gene-expression processes (m=34); NOT a single pooled threshold)")
for target, row in partial_df.iterrows():
    a = target_alpha[target]
    sig_raw = "*" if row['p_raw'] < a else " "
    sig_th = "*" if row['p_partial_thickness'] < a else " "
    sig_ar = "*" if row['p_partial_area'] < a else " "
    sig_geo = "*" if row['p_partial_geometry'] < a else " "
    sig_cur = "*" if row['p_current'] < a else " "
    print(f"{target:20s} | P raw: rho={row['rho_raw']:+.3f}, p={row['p_raw']:.4f}{sig_raw} "
          f"| thick: rho={row['rho_partial_thickness']:+.3f}, p={row['p_partial_thickness']:.4f}{sig_th} "
          f"| area: rho={row['rho_partial_area']:+.3f}, p={row['p_partial_area']:.4f}{sig_ar} "
          f"| geometry (1/[L*A]): rho={row['rho_partial_geometry']:+.3f}, p={row['p_partial_geometry']:.4f}{sig_geo} "
          f"| I_dip raw: rho={row['rho_current']:+.3f}, p={row['p_current']:.4f}{sig_cur}")
print("(* = significant at the Bonferroni threshold matching that target's own family, per above)")

In [ ]:
# Supplementary figure: raw vs. geometry-adjusted |rho| restricted to the targets actually
# highlighted in the main text -- the auto-selected top-2 ketone-body processes
# (top2_processes) and CMrO2 (oxygen metabolism) -- so a reader can see directly whether each
# headline correlation survives controlling for cortical geometry, and whether it's driven
# more by thickness, area, the exact 1/(L*A) term that appears in P's own formula, or the
# current-dipole term itself. CMrGlu is omitted here since it isn't a significant main-text
# finding.
highlight_targets = list(dict.fromkeys(top2_processes + ['cmro2']))
plot_df = partial_df.loc[highlight_targets]

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(highlight_targets))
width = 0.16
bars_raw = ax.bar(x - 2 * width, plot_df['rho_raw'].abs(), width, label='Raw P (zero-order)')
bars_th = ax.bar(x - width, plot_df['rho_partial_thickness'].abs(), width, label='Partial: thickness')
bars_ar = ax.bar(x, plot_df['rho_partial_area'].abs(), width, label='Partial: area')
bars_geo = ax.bar(x + width, plot_df['rho_partial_geometry'].abs(), width, label='Partial: geometry (1/[L*A])')
bars_cur = ax.bar(x + 2 * width, plot_df['rho_current'].abs(), width, label='Raw I_dip (current dipole)')

# Significance flagged per-target against ITS OWN family's Bonferroni threshold (alpha_bonf_met
# for cmrglu/cmro2, alpha_bonf_proc for gene-expression processes) -- not one pooled threshold
# across both types, matching Fig. 4/5's convention (see markdown cell above).
bar_pcol_pairs = [
    (bars_raw, 'p_raw'), (bars_th, 'p_partial_thickness'), (bars_ar, 'p_partial_area'),
    (bars_geo, 'p_partial_geometry'), (bars_cur, 'p_current'),
]
for bars, pcol in bar_pcol_pairs:
    for bar, target in zip(bars, highlight_targets):
        if plot_df.loc[target, pcol] < target_alpha[target]:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, '*',
                    ha='center', va='bottom', fontsize=14)

ax.set_xticks(x)
ax.set_xticklabels(highlight_targets, rotation=20, ha='right')
ax.set_ylabel('Spearman |rho|')
ax.set_title('Primary dipole P vs. PET/transcriptomic measures:\nraw, geometry-partialled, and current-dipole-only correlations')
ax.legend(ncol=2, fontsize=11)
plt.tight_layout()
plt.savefig(output_dir + 'supp_partial_correlations_geometry.png', transparent=False)
plt.show()

print(f"(* = significant at the Bonferroni threshold matching that target's own family: "
      f"alpha={alpha_bonf_met:.5f} for cmrglu/cmro2 (m=2, same as Fig. 4), "
      f"alpha={alpha_bonf_proc:.5f} for gene-expression processes (m=34, same as Fig. 5) -- "
      f"NOT a single pooled threshold across both types. Full 36-target table saved to "
      f"supplementary_partial_correlations_power.csv)")

## Optional: switch to secondary power instead of primary

Everything above uses **primary** dipole power (I_dip^2 * cortical-column resistance,
32-subject cohort) as the main data source. **Secondary** power (DUNEuro FEM
volume-conductor power dissipation) is a separate, smaller-cohort (8-subject) analysis --
see `meg_power/secondary_p/eeg_dipole_power_computation.py` +
`parcellate_central_surface_s600_secondary.m`. It has no current-dipole ("i") analog of its
own (see `compare_pri_sec_dip_p.m`), so only `meg_rgn_p`'s power column can be swapped --
`meg_rgn_i` has no secondary equivalent and stays primary-current in every comparison.

To re-run the notebook against secondary power instead: uncomment the block below (after
`parcellate_central_surface_s600_secondary.m` has produced
`s600_p_sec_2min_rest_subavg_snr3.csv`) and re-run every cell from here on -- all the
plotting/null-model code above is written generically against `meg_rgn_p`/`meg_rgn_i`, so
nothing else needs to change.

In [ ]:
# SECONDARY_CSV = '/export02/data/vikramn/hbm_manuscript_code/outputs/secondary_p/cent_surf_fem_fwd/s600_p_sec_2min_rest_subavg_snr3.csv'
# sec_p = pd.read_csv(SECONDARY_CSV).rename(columns={'value': 'mean'})
# meg_rgn_p = sec_p.merge(meg_met[['region', 'cmrglu', 'cmro2']], on='region', how='inner').merge(
#     met_df, on='region', how='inner')
# print(f'Secondary power: {len(meg_rgn_p)} regions (8-subject cohort, vs. {len(meg_met)} for primary).')
#
# # Re-run e.g. the whole-brain CMrGlu/CMrO2 barplot cell (or any other cell above) after this
# # to see it recomputed against secondary power. meg_rgn_i (current dipole) is left untouched
# # since secondary power has no current-dipole analog.